In [167]:
import pandas as pd
import altair as alt

In [168]:
myJekyllDir = "C:/Users/larat/Desktop/LaraTerpetschnig.github.io/assets/json/"

# Plot 1

In [169]:
df = pd.read_csv("https://raw.githubusercontent.com/LaraTerpetschnig/LaraTerpetschnig.github.io/refs/heads/main/data/student-por.csv", sep=';')
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [170]:
data = pd.pivot_table(df, values='G3', index='Walc', columns='Dalc', aggfunc='mean') 
data

Dalc,1,2,3,4,5
Walc,,,,,
1,12.369295,11.333333,18.000000,9.000000,11.0
2,12.654867,11.470588,9.000000,0.000000,10.0
3,11.671875,12.093023,10.888889,8.750000,NaN
4,11.857143,10.558824,11.100000,9.400000,NaN
5,11.400000,10.285714,11.000000,10.166667,10.2


In [171]:
data = data.reset_index().rename_axis(None, axis=1)
data

,Walc,1,2,3,4,5
0,1,12.369295,11.333333,18.000000,9.000000,11.0
1,2,12.654867,11.470588,9.000000,0.000000,10.0
2,3,11.671875,12.093023,10.888889,8.750000,NaN
3,4,11.857143,10.558824,11.100000,9.400000,NaN
4,5,11.400000,10.285714,11.000000,10.166667,10.2


In [172]:
data_melted = data.melt(id_vars=['Walc'], value_vars=[1, 2, 3, 4, 5])
data_melted.rename({'variable':'Dalc'}, axis=1, inplace=True)
data_melted

,Walc,Dalc,value
0,1,1,12.369295
1,2,1,12.654867
2,3,1,11.671875
3,4,1,11.857143
4,5,1,11.400000
5,1,2,11.333333
6,2,2,11.470588
7,3,2,12.093023
8,4,2,10.558824
9,5,2,10.285714


In [173]:
data_melted['value'].min(), data_melted['value'].max()

(0.0, 18.0)

In [174]:
title = alt.TitleParams("Final Grades based on Weekday and Weekend Drinking Levels", anchor='middle')

select = alt.selection_point(fields=['Walc', 'Dalc'])

heatmap = alt.Chart(
    df.pivot_table(values='G3', index='Walc', columns='Dalc', aggfunc='mean') \
                    .reset_index().rename_axis(None, axis=1) \
                    .melt(id_vars=['Walc'], value_vars=[1, 2, 3, 4, 5]) \
                    .rename({'variable':'Dalc', 'value':'Grade'}, axis=1),
    height=250,
    width=250,
).mark_rect().encode(
    x=alt.X('Walc:O').title("Weekend Drinking"),
    y=alt.Y('Dalc:O', scale=alt.Scale(reverse=True)).title("Weekday Drinking"),
    color=alt.Color('Grade').scale(scheme='greens'),
    opacity=alt.when(select).then(alt.value(1)).otherwise(alt.value(0.4)),
).add_params(
    select
)

hist = alt.Chart(df).mark_bar().encode(
    x=alt.X("G3", bin=True, title='Final Grade'),
    y=alt.Y('count(G3):Q', title='Frequency'),
    color=alt.value("#DA291C")
).transform_filter(
    select
)

interactive_heatmap = alt.hconcat(hist, heatmap, title=title)
interactive_heatmap

alt.HConcatChart(...)

In [164]:
interactive_heatmap.save(myJekyllDir + 'final_project_interactive_heatmap.json')

# Plot 2

In [75]:
df2 = pd.read_csv("https://raw.githubusercontent.com/LaraTerpetschnig/LaraTerpetschnig.github.io/refs/heads/main/data/Portugal_Alcohol_Consumption_15_to_24.csv")
df2.head()

,Year,Every day,Every week,Every month,Less than once a month,Not in the last 12 months,Never
0,2014,0.6,19.9,27.1,20.4,4.0,27.9
1,2019,1.2,21.2,23.5,16.3,4.1,33.8


In [76]:
df2_melted = df2.melt(id_vars=['Year'], value_vars=['Every day', 'Every week', 'Every month', 'Less than once a month',
                                       'Not in the last 12 months', 'Never'])
df2_melted = df2_melted.rename({'value':'Percentage', 'variable':'Frequency of Consumption'}, axis=1)
df2_melted

,Year,Frequency of Consumption,Percentage
0,2014,Every day,0.6
1,2019,Every day,1.2
2,2014,Every week,19.9
3,2019,Every week,21.2
4,2014,Every month,27.1
5,2019,Every month,23.5
6,2014,Less than once a month,20.4
7,2019,Less than once a month,16.3
8,2014,Not in the last 12 months,4.0
9,2019,Not in the last 12 months,4.1


In [82]:
title = alt.TitleParams("Frequency of Alcohol Consumption for Portuguese Citizens Ages 15 - 24", anchor='middle')

grouped_bars = alt.Chart(df2_melted, title=title).mark_bar().encode(
    x=alt.X('Year:O', title=''),
    y='Percentage:Q',
    color='Year:O',
    column=alt.Column('Frequency of Consumption:N', title='', sort=['Every day', 'Every week', 
                                                                    'Every month', 'Less than once a month',
                                                                    'Not in the last 12 months', 'Never'])
).properties(
    width=100
)

grouped_bars

alt.Chart(...)

In [83]:
grouped_bars.save(myJekyllDir + 'final_project_grouped_bars.json')

# Plot 3

In [91]:
df3 = pd.read_csv("https://raw.githubusercontent.com/LaraTerpetschnig/LaraTerpetschnig.github.io/refs/heads/main/data/EU_Alcohol_Consumption_2019.csv")
df3

,Country,Every day,Every week,Every month,Less than once a month,Not in the last 12 months,Never
0,Germany,0.4,27.3,39.4,15.9,1.4,15.6
1,Spain,0.0,26.6,29.4,13.2,5.3,25.4
2,France,1.7,24.7,31.8,18.3,2.8,20.7
3,Italy,1.0,27.3,22.4,9.6,1.3,38.4
4,Poland,0.0,11.7,30.3,27.4,10.7,19.9
5,Portugal,0.0,19.4,29.3,21.8,1.6,27.9


In [92]:
df3_melted = df3.melt(id_vars=['Country'], value_vars=['Every day', 'Every week', 'Every month', 'Less than once a month',
                                       'Not in the last 12 months', 'Never'])
df3_melted

,Country,variable,value
0,Germany,Every day,0.4
1,Spain,Every day,0.0
2,France,Every day,1.7
3,Italy,Every day,1.0
4,Poland,Every day,0.0
5,Portugal,Every day,0.0
6,Germany,Every week,27.3
7,Spain,Every week,26.6
8,France,Every week,24.7
9,Italy,Every week,27.3


In [134]:
selection_legend = alt.selection_point(fields=['variable'], bind='legend')

stacked_bars = alt.Chart(df3_melted, title='Comparison of Top 5 EU Countries and Portugal').mark_bar().transform_calculate(
    variable_order=f"if({selection_legend.name}.variable && indexof({selection_legend.name}.variable, datum.variable) !== -1, 0, 1)"
).encode(
    x=alt.X('sum(value):Q', title='Percentage', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('Country:N').sort(['Germany', 'France', 'Italy', 'Spain', 'Poland', 'Portugal']),
    color=alt.Color('variable:N', legend=alt.Legend(title="Frequency of Consumption"), 
                    sort=['Every day', 'Every week', 'Every month', 'Less than once a month',
                                       'Not in the last 12 months', 'Never']),
    order='variable_order:N',
    opacity=alt.when(selection_legend).then(alt.value(0.9)).otherwise(alt.value(0.2))
).add_params(
    selection_legend
).properties(
    width=600
)

stacked_bars

alt.Chart(...)

In [135]:
stacked_bars.save(myJekyllDir + 'final_project_stacked_bars.json')